# Optuna sweep over structural ISP knobs (Colab)

Thin Colab wrapper around `scripts/run_optuna_isp_knobs.py`. Runs an Optuna search over the non-differentiable structural ISP knobs while the residual CNN and the trained ISP weights stay frozen.

Each trial samples one combination of structural knobs, builds a fresh `ISPPipeline`, loads the trained ISP weight tensors and the CNN from the supplied end-to-end checkpoint, and evaluates the resulting pipeline on the per-scene mini-val split. The objective is the mean-over-scenes composite `VIF_norm + a*NRQM_norm + b*UNIQUE_norm`.

## Required Google Drive files

Create this folder:

`MyDrive/ISP_colab/optuna_isp_knobs/`

Upload these two files there:

- `optuna_tuning_bundle.zip` 
- `e2e_best.pth` (the trained ISP+CNN checkpoint)

Outputs are written to:

`MyDrive/ISP_colab/optuna_isp_knobs_outputs/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Configuration

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ISP_colab')
DRIVE_INPUT = DRIVE_ROOT / 'optuna_isp_knobs'
DRIVE_OUTPUT = DRIVE_ROOT / 'optuna_isp_knobs_outputs'

BUNDLE_ZIP = DRIVE_INPUT / 'optuna_tuning_bundle.zip'
CKPT_FILE  = DRIVE_INPUT / 'e2e_best.pth'

LOCAL_REPO = Path('/content/ISP')
LOCAL_CKPT = LOCAL_REPO / 'artifacts' / 'checkpoints' / 'e2e_quality' / 'e2e_best.pth'

N_TRIALS = 100
N_STARTUP_TRIALS = 16
EVAL_MAX_FRAMES = 3
SEED = 42

RESET_OUTPUTS = False
RESUME_STUDY = True

print(f'Drive input:  {DRIVE_INPUT}')
print(f'Drive output: {DRIVE_OUTPUT}')
print(f'Local repo:   {LOCAL_REPO}')

## 2. Check GPU

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Please switch Colab runtime to GPU before starting Optuna tuning.')

## 3. Install dependencies

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

%pip -q install optuna pyiqa

## 4. Prepare local workspace

In [ ]:
import shutil
import time
import zipfile

def required(path):
    path = Path(path)
    if not path.exists():
        parent = path.parent
        available = sorted(p.name for p in parent.iterdir()) if parent.exists() else []
        msg = f'Missing required file: {path}'
        if available:
            msg += '\n\nFiles currently in that folder:\n' + '\n'.join(f'  - {n}' for n in available)
        raise FileNotFoundError(msg)
    return path

def required_one(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    listed = ', '.join(str(Path(p)) for p in paths)
    raise FileNotFoundError(f'Missing required file (looked for one of: {listed})')

BUNDLE_ZIP = DRIVE_INPUT / 'optuna_tuning_bundle.zip'

if LOCAL_REPO.exists():
    print(f'Removing old local workspace: {LOCAL_REPO}')
    shutil.rmtree(LOCAL_REPO)
LOCAL_REPO.mkdir(parents=True, exist_ok=True)

if RESET_OUTPUTS and DRIVE_OUTPUT.exists():
    assert DRIVE_OUTPUT.name == 'optuna_isp_knobs_outputs'
    print(f'Removing old Optuna outputs: {DRIVE_OUTPUT}')
    shutil.rmtree(DRIVE_OUTPUT)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

required(CKPT_FILE)

print(f'Unzipping {BUNDLE_ZIP.name}...', end=' ', flush=True)
t0 = time.time()
with zipfile.ZipFile(BUNDLE_ZIP, 'r') as zf:
    zf.extractall(LOCAL_REPO)
print(f'done ({time.time() - t0:.1f}s)')

LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT_FILE, LOCAL_CKPT)
size_mb = LOCAL_CKPT.stat().st_size / 1024 ** 2
print(f'Copied checkpoint: {LOCAL_CKPT} ({size_mb:.2f} MB)')

checks = [
    LOCAL_REPO / 'scripts' / 'run_optuna_isp_knobs.py',
    LOCAL_REPO / 'isp' / 'pipeline' / 'pipeline.py',
    LOCAL_REPO / 'metrics' / 'vif.py',
    LOCAL_REPO / 'data' / 'imx623.toml',
    LOCAL_REPO / 'dataset' / 'splits_mini.json',
    LOCAL_REPO / 'artifacts' / 'baselines' / 'norm_weights.json',
    LOCAL_CKPT,
]
for name in ['day_val_mini.bin', 'day_val_mini.yuv', 'night_val_mini.bin', 'night_val_mini.yuv']:
    checks.append(LOCAL_REPO / 'data' / name)
for path in checks:
    print(f'{path.relative_to(LOCAL_REPO)}: {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

bundle_markers = [
    (LOCAL_REPO / 'scripts' / 'run_optuna_isp_knobs.py', 'suggest_categorical(name, list(low))'),
    (LOCAL_REPO / 'scripts' / 'run_optuna_isp_knobs.py', '("ltm_downsample",       "choice", (0.25, 0.5, 0.75, 1.0), None, False)'),
    (LOCAL_REPO / 'scripts' / 'run_optuna_isp_knobs.py', '("sharp_radius",         "choice", (0.5, 1.0, 1.5, 2.0, 2.5, 3.0), None, False)'),
]
for path, marker in bundle_markers:
    text = path.read_text(encoding='utf-8')
    if marker not in text:
        raise RuntimeError(
            f'Stale bundle detected: {path.name} does not contain marker {marker!r}. '
            'Upload optuna_tuning_bundle_fixed.zip or replace optuna_tuning_bundle.zip with the rebuilt bundle.'
        )
print('Bundle self-check: OK (discrete Optuna choices present)')

print('\nAll Optuna inputs are ready.')

## 5. Resume status

In [ ]:
import sqlite3

study_db = DRIVE_OUTPUT / 'optuna_isp_knobs.db'

def study_counts(db_path):
    counts = {'total': 0}
    if not db_path.exists():
        return counts
    with sqlite3.connect(db_path) as con:
        rows = con.execute('select state, count(*) from trials group by state').fetchall()
    for state, n in rows:
        counts[str(state).lower()] = int(n)
        counts['total'] += int(n)
    return counts

counts = study_counts(study_db)
existing = counts['total'] if (study_db.exists() and RESUME_STUDY and not RESET_OUTPUTS) else 0
TRIALS_TO_RUN = max(0, int(N_TRIALS) - existing)

print(f'Study DB:               {study_db}')
print(f'Existing trials:        {existing}  {counts}')
print(f'Target N_TRIALS:        {N_TRIALS}')
print(f'Trials to add this run: {TRIALS_TO_RUN}')

## 6. Run Optuna sweep

In [ ]:
import subprocess
import sys

if 'TRIALS_TO_RUN' not in globals():
    TRIALS_TO_RUN = int(N_TRIALS)

cmd = [
    sys.executable, '-B', 'scripts/run_optuna_isp_knobs.py',
    '--device', 'cuda',
    '--ckpt', str(LOCAL_CKPT.relative_to(LOCAL_REPO)),
    '--config', 'data/imx623.toml',
    '--splits-json', 'dataset/splits_mini.json',
    '--norm-weights', 'artifacts/baselines/norm_weights.json',
    '--output-dir', str(DRIVE_OUTPUT),
    '--n-trials', str(TRIALS_TO_RUN),
    '--n-startup-trials', str(N_STARTUP_TRIALS),
    '--eval-max-frames', str(EVAL_MAX_FRAMES),
    '--seed', str(SEED),
]
if RESUME_STUDY:
    cmd += ['--resume-study']
if RESET_OUTPUTS:
    cmd += ['--reset-outputs']

if TRIALS_TO_RUN == 0:
    print('Trial budget already met; running reporting only.')
print('Running:')
print(' '.join(cmd))
print()

proc = subprocess.Popen(
    cmd, cwd=str(LOCAL_REPO),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
finally:
    if proc.poll() is None:
        proc.terminate()

print(f'\n[subprocess exit={ret}]')
if ret != 0:
    raise RuntimeError(f'Optuna script exited with code {ret}')

## 7. Inspect best trial

In [ ]:
import json

best_path = DRIVE_OUTPUT / 'optuna_best_params.json'
if best_path.exists():
    with open(best_path, 'r', encoding='utf-8') as f:
        best = json.load(f)
    print(f'Best trial:    {best["best_trial"]}')
    print(f'Best objective: {best["best_value"]:.6f}')
    attrs = best.get('best_user_attrs', {})
    print(f'avg VIF:        {attrs.get("avg_vif")}')
    print(f'avg NRQM:       {attrs.get("avg_nrqm")}')
    print(f'avg UNIQUE:     {attrs.get("avg_unique")}')
    print('\nBest params:')
    print(json.dumps(best.get('best_params', {}), indent=2))
else:
    print(f'No best params file at {best_path} yet.')

## 8. Plots

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for name in ('optuna_history.png', 'optuna_importance.png'):
    path = DRIVE_OUTPUT / name
    if path.exists():
        plt.figure(figsize=(8, 5))
        plt.imshow(Image.open(path))
        plt.title(name)
        plt.axis('off')
        plt.show()
    else:
        print(f'Missing: {path}')